# Модель кредитного скоринга

**Цель:** прогноз дефолта малого бизнеса в течение 12 месяцев (`default_12m`).

**Данные:** 20 000 синтетических заявок, 15 признаков.

## Поля модели

| Поле | Описание |
|---|---|
| `amount` | Сумма кредита |
| `term_months` | Срок кредита в месяцах |
| `purpose` | Цель кредита |
| `legal_form` | Организационно-правовая форма (ИП/ООО) |
| `has_collateral` | Наличие залога |
| `collateral_type` | Тип залога |
| `business_age_months` | Возраст бизнеса в месяцах |
| `annual_revenue` | Годовая выручка |
| `monthly_turnover` | Ежемесячный оборот |
| `monthly_expenses` | Ежемесячные расходы |
| `existing_debt` | Текущая задолженность |
| `monthly_debt_payments` | Ежемесячные платежи по долгам |
| `overdue_days` | Количество дней просрочки |
| `collateral_value` | Стоимость залога |
| `industry` | Отрасль бизнеса |

**Целевая переменная:** `default_12m` — факт дефолта в течение 12 месяцев (1 — дефолт, 0 — нет).

## Генерация и подготовка данных

Сгенерированы 20 000 синтетических заявок с финансовыми показателями, долговой нагрузкой, просрочками и залогом. Целевая переменная формируется вероятностно на основе заданных правил риска. Данные разделены на обучающую и тестовую выборки в пропорции 80/20.

## Технологии

- **Python** — основной язык.
- **Pandas, NumPy** — генерация и обработка данных.
- **Scikit-learn** — предобработка признаков, разделение выборок и обучение моделей.
- **CatBoost** — обучение лучшей модели.
- **LightGBM** — дополнительный алгоритм для сравнения.
- **Matplotlib / Seaborn** — визуализация, если использовались в ноутбуке.

## Сравнение алгоритмов

Проверены Logistic Regression, Random Forest, CatBoost и LightGBM.

**Лучшая модель:** CatBoost.

Результаты на тестовой выборке:
- ROC-AUC: **0,754**
- Precision: **0,612**
- Recall: **0,675**
- F1-score: **0,642**

**Ограничение:** модель обучена на синтетических данных. Её качество на реальных заёмщиках не проверено; результаты не отражают подтверждённую точность банковского скоринга.

In [ ]:
!pip install -q catboost lightgbm

In [11]:
import glob
import zipfile
import os

zips = glob.glob("/content/*recommende*.zip")
print("Найденные архивы:", zips)

if zips:
    with zipfile.ZipFile(zips[-1]) as z:
        z.extractall("/content/ai-credit-assistant")
    print("Архив распакован")
else:
    print("ZIP не найден — проверь имя файла")

Найденные архивы: ['/content/ai-credit-assistant-recommender.zip']
Архив распакован


In [12]:
!find /content/ai-credit-assistant -name "recommender.py" -o -name "products.json"

/content/ai-credit-assistant/ai-credit-assistant-recommender/scoring/recommender/products.json
/content/ai-credit-assistant/ai-credit-assistant-recommender/scoring/recommender/recommender.py


In [19]:

import numpy as np
import pandas as pd

# Берём категории из существующего проекта
try:
    from recommender import PURPOSES
    purposes = list(PURPOSES)
except ImportError:
    purposes = [
        "working_capital",
        "equipment",
        "expansion",
        "refinancing",
    ]

rng = np.random.default_rng(42)
N = 20_000

# 1. Генерируем заявку
df = pd.DataFrame({
    "amount": np.clip(
        rng.lognormal(np.log(2_000_000), 1.0, N),
        50_000, 60_000_000
    ).astype(int),

    "term_months": rng.choice(
        [6, 12, 18, 24, 36, 48, 60, 84, 120], N
    ),

    "purpose": rng.choice(purposes, N),

    "legal_form": rng.choice(
        ["IP", "OOO"], N, p=[0.55, 0.45]
    ),

    "has_collateral": rng.random(N) < 0.4,

    "business_age_months": np.clip(
        rng.gamma(2.0, 20, N).astype(int), 1, 240
    ),
})

df["collateral_type"] = np.where(
    df["has_collateral"],
    rng.choice(
        ["real_estate", "vehicle", "equipment"], N
    ),
    "none",
)

# 2. Генерируем финансовые данные и данные БКИ
df["annual_revenue"] = rng.lognormal(
    np.log(15_000_000), 0.8, N
).round(0)

df["monthly_turnover"] = (
    df["annual_revenue"] / 12
    * rng.lognormal(0, 0.12, N)
).round(0)

df["monthly_expenses"] = (
    df["monthly_turnover"] * rng.uniform(0.45, 0.92, N)
).round(0)

df["existing_debt"] = (
    df["annual_revenue"] * rng.uniform(0, 0.5, N)
).round(0)

df["monthly_debt_payments"] = (
    df["existing_debt"] * rng.uniform(0.02, 0.10, N)
).round(0)

df["overdue_days"] = np.clip(
    rng.exponential(8, N).astype(int), 0, 120
)

df["collateral_value"] = np.where(
    df["has_collateral"],
    (
        df["amount"] * rng.uniform(0.5, 1.5, N)
    ).round(0),
    0,
)

df["industry"] = rng.choice(
    ["trade", "services", "manufacturing",
     "construction", "it"],
    N,
    p=[0.30, 0.30, 0.12, 0.18, 0.10],
)

# 3. Рассчитываем финансовые отношения
expense_ratio = (
    df["monthly_expenses"] /
    df["monthly_turnover"].clip(lower=1)
)

debt_ratio = (
    df["monthly_debt_payments"] /
    df["monthly_turnover"].clip(lower=1)
)

loan_ratio = (
    df["amount"] /
    df["annual_revenue"].clip(lower=1)
)

collateral_coverage = (
    df["collateral_value"] /
    df["amount"].clip(lower=1)
)

# Условное влияние цели кредита.
# Неизвестные категории получают нейтральный коэффициент.
purpose_risk = df["purpose"].map({
    "working_capital": 0.0,
    "equipment": -0.15,
    "expansion": 0.15,
    "refinancing": 0.10,
}).fillna(0.0)

# 4. Генерируем вероятность дефолта
# Коэффициенты являются гипотезами для синтетического прототипа.
logit = (
    -3.0
    + 1.8 * expense_ratio
    + 5.0 * debt_ratio
    + 1.5 * np.log1p(loan_ratio)
    + 0.035 * df["overdue_days"]
    - 0.012 * df["business_age_months"]
    - 0.5 * df["has_collateral"].astype(int)
    - 0.5 * np.clip(collateral_coverage, 0, 1.5)
    + 0.015 * df["term_months"]
    + purpose_risk
    + 0.3 * (df["industry"] == "construction")
)

pd_true = 1 / (1 + np.exp(-logit))

# Случайно генерируем факт дефолта на основе вероятности
df["default_12m"] = rng.binomial(1, pd_true)

# 5. Финальная схема датасета
FEATURES = [
    "amount",
    "term_months",
    "purpose",
    "legal_form",
    "has_collateral",
    "collateral_type",
    "business_age_months",
    "annual_revenue",
    "monthly_turnover",
    "monthly_expenses",
    "existing_debt",
    "monthly_debt_payments",
    "overdue_days",
    "collateral_value",
    "industry",
]

train = df[FEATURES + ["default_12m"]].copy()
train["has_collateral"] = train["has_collateral"].astype(int)

# Сохраняем только признаки и целевую переменную
train.to_csv("train.csv", index=False)

# 6. Проверяем результат
print("Размер датасета:", train.shape)
print("Доля дефолтов:", round(train["default_12m"].mean(), 3))

print("\nРаспределение классов:")
print(train["default_12m"].value_counts(normalize=True).round(3))

print("\nСредние финансовые отношения по классам:")
check = pd.DataFrame({
    "expense_ratio": expense_ratio,
    "debt_ratio": debt_ratio,
    "loan_ratio": loan_ratio,
    "collateral_coverage": collateral_coverage,
    "default_12m": df["default_12m"],
})
display(check.groupby("default_12m").mean().round(3))

print("\nКолонки:", train.columns.tolist())
display(train.head())
print("\nСохранено в train.csv")


Размер датасета: (20000, 16)
Доля дефолтов: 0.417

Распределение классов:
default_12m
0    0.583
1    0.417
Name: proportion, dtype: float64

Средние финансовые отношения по классам:


,expense_ratio,debt_ratio,loan_ratio,collateral_coverage
default_12m,,,,
0,0.674,0.151,0.237,0.477
1,0.696,0.226,0.410,0.271



Колонки: ['amount', 'term_months', 'purpose', 'legal_form', 'has_collateral', 'collateral_type', 'business_age_months', 'annual_revenue', 'monthly_turnover', 'monthly_expenses', 'existing_debt', 'monthly_debt_payments', 'overdue_days', 'collateral_value', 'industry', 'default_12m']


,amount,term_months,purpose,legal_form,has_collateral,collateral_type,business_age_months,annual_revenue,monthly_turnover,monthly_expenses,existing_debt,monthly_debt_payments,overdue_days,collateral_value,industry,default_12m
0,2712482,24,refinancing,OOO,1,real_estate,22,2048211.0,184422.0,89395.0,232317.0,7420.0,11,3081243.0,manufacturing,0
1,706920,24,refinancing,OOO,0,none,4,22041759.0,1649682.0,1505986.0,2914857.0,116608.0,1,0.0,manufacturing,0
2,4235910,6,expansion,IP,1,equipment,50,11242917.0,659281.0,403816.0,2866226.0,228540.0,2,2423579.0,services,0
3,5122854,120,refinancing,IP,0,none,108,22013674.0,1482965.0,779854.0,7861714.0,726808.0,1,0.0,services,0
4,284253,84,refinancing,OOO,1,equipment,73,21852526.0,1486559.0,846958.0,8419399.0,527860.0,2,303237.0,manufacturing,0



Сохранено в train.csv


In [20]:

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# 1. Загружаем датасет
df = pd.read_csv("train.csv")

TARGET = "default_12m"

FEATURES = [
    "amount",
    "term_months",
    "purpose",
    "legal_form",
    "has_collateral",
    "collateral_type",
    "business_age_months",
    "annual_revenue",
    "monthly_turnover",
    "monthly_expenses",
    "existing_debt",
    "monthly_debt_payments",
    "overdue_days",
    "collateral_value",
    "industry",
]

X = df[FEATURES].copy()
y = df[TARGET].astype(int)

# Приводим булево поле к числовому формату
X["has_collateral"] = X["has_collateral"].astype(int)

# 2. Разделяем данные ДО обучения преобразований
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 3. Определяем типы признаков
CATEGORICAL = [
    "purpose",
    "legal_form",
    "collateral_type",
    "industry",
]

NUMERIC = [
    col for col in FEATURES
    if col not in CATEGORICAL
]

# 4. Обработка числовых признаков:
# пропуски заполняем медианой, затем стандартизируем
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# 5. Обработка категориальных признаков:
# пропуски заполняем самым частым значением,
# категории превращаем в бинарные колонки
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

# 6. Объединяем преобразования
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, NUMERIC),
    ("cat", categorical_pipeline, CATEGORICAL),
])

In [21]:
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

# Сравниваем 4 модели
models = {
    "Logistic Regression": Pipeline([
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            max_iter=2000,
            class_weight="balanced"
        ))
    ]),

    "Random Forest": Pipeline([
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=200,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ]),

    "CatBoost": CatBoostClassifier(
        iterations=500,
        depth=6,
        learning_rate=0.05,
        verbose=False,
        random_seed=42,
        auto_class_weights="Balanced"
    ),

    "LightGBM": Pipeline([
        ("preprocessor", preprocessor),
        ("model", LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=20,
            class_weight="balanced",
            verbosity=-1,
            random_state=42
        ))
    ])
}

results = {}
trained_models = {}

for name, model in models.items():
    # CatBoost получает исходные категории,
    # остальные модели — обработанные признаки
    if name == "CatBoost":
        model.fit(X_train, y_train, cat_features=[
            "purpose", "legal_form",
            "collateral_type", "industry"
        ])
    else:
        model.fit(X_train, y_train)

    pd_pred = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, pd_pred)

    results[name] = auc
    trained_models[name] = model

    print(f"{name}: ROC-AUC = {auc:.3f}")

# Сравнение результатов
print("\nРезультаты по ROC-AUC:")
for name, auc in sorted(
    results.items(), key=lambda item: item[1], reverse=True
):
    print(f"{name}: {auc:.3f}")

# Сохраняем модель с лучшим ROC-AUC
best_name = max(results, key=results.get)
best_model = trained_models[best_name]

joblib.dump(best_model, "risk_model.joblib")

print(f"\nЛучшая модель: {best_name}")
print("Сохранена в risk_model.joblib")


Logistic Regression: ROC-AUC = 0.731
Random Forest: ROC-AUC = 0.741
CatBoost: ROC-AUC = 0.754
LightGBM: ROC-AUC = 0.749

Результаты по ROC-AUC:
CatBoost: 0.754
LightGBM: 0.749
Random Forest: 0.741
Logistic Regression: 0.731

Лучшая модель: CatBoost
Сохранена в risk_model.joblib


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [22]:

import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
    brier_score_loss,
    log_loss,
)

# 1. Получаем вероятность дефолта
pd_pred = best_model.predict_proba(X_test)[:, 1]

# Порог 0.5 используется только для первичной оценки
threshold = 0.5
y_pred = (pd_pred >= threshold).astype(int)

# 2. Считаем метрики
metrics = {
    "ROC-AUC": roc_auc_score(y_test, pd_pred),
    "PR-AUC (Average Precision)": average_precision_score(
        y_test, pd_pred
    ),
    "Accuracy": accuracy_score(y_test, y_pred),
    "Balanced Accuracy": balanced_accuracy_score(
        y_test, y_pred
    ),
    "Precision (default)": precision_score(
        y_test, y_pred, zero_division=0
    ),
    "Recall (default)": recall_score(
        y_test, y_pred, zero_division=0
    ),
    "F1 (default)": f1_score(
        y_test, y_pred, zero_division=0
    ),
    "Brier Score": brier_score_loss(y_test, pd_pred),
    "Log Loss": log_loss(y_test, pd_pred),
}

print("Модель:", best_name)
print("Порог классификации:", threshold)
display(pd.DataFrame(
    {"Metric": metrics.keys(), "Value": metrics.values()}
).round(4))

# 3. Подробный отчёт по классам
print("\nClassification report:")
print(classification_report(
    y_test, y_pred,
    target_names=["Нет дефолта", "Дефолт"],
    digits=3,
    zero_division=0
))

# 4. Матрица ошибок
cm = confusion_matrix(y_test, y_pred)
display(pd.DataFrame(
    cm,
    index=["Факт: нет дефолта", "Факт: дефолт"],
    columns=["Прогноз: нет дефолта", "Прогноз: дефолт"]
))


Модель: CatBoost
Порог классификации: 0.5


,Metric,Value
0,ROC-AUC,0.7540
1,PR-AUC (Average Precision),0.6940
2,Accuracy,0.6855
3,Balanced Accuracy,0.6840
4,Precision (default),0.6116
5,Recall (default),0.6747
6,F1 (default),0.6416
7,Brier Score,0.2002
8,Log Loss,0.5834



Classification report:
              precision    recall  f1-score   support

 Нет дефолта      0.748     0.693     0.720      2331
      Дефолт      0.612     0.675     0.642      1669

    accuracy                          0.685      4000
   macro avg      0.680     0.684     0.681      4000
weighted avg      0.691     0.685     0.687      4000



,Прогноз: нет дефолта,Прогноз: дефолт
Факт: нет дефолта,1616,715
Факт: дефолт,543,1126
